# Análisis exploratorio del dataset preparado

Este notebook analiza el dataset generado durante la fase de preparación. La unidad de observación es el cliente y cada registro contiene la variable objetivo `churn`, la partición original y las 176 variables predictoras seleccionadas.

El objetivo de esta fase es comprender la estructura y las características estadísticas de los datos antes de iniciar el modelado. Para ello, se estudian la distribución de la variable objetivo, la separación entre los conjuntos de entrenamiento y prueba, la variabilidad de las variables predictoras, la presencia de valores extremos y sus relaciones más relevantes con el `churn`.

El análisis exploratorio se realiza respetando las particiones originales. Las decisiones metodológicas y la selección de modelos se basarán exclusivamente en el conjunto de entrenamiento, mientras que el conjunto de prueba permanecerá separado hasta la evaluación final. Este enfoque reduce el riesgo de fugas de información y de obtener estimaciones de rendimiento excesivamente optimistas.

Durante esta fase no se modifican los archivos originales ni las matrices preparadas. Los resultados derivados se almacenan por separado en la carpeta `resultados`, manteniendo la trazabilidad entre los datos de origen, los análisis realizados y las conclusiones obtenidas.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display


# ---------------------------------------------------------
# Definición de las rutas del proyecto
# ---------------------------------------------------------

RUTA_CUADERNO = Path.cwd()

if RUTA_CUADERNO.name == "notebooks":
    RUTA_PROYECTO = RUTA_CUADERNO.parent
else:
    RUTA_PROYECTO = RUTA_CUADERNO

RUTA_DATOS = RUTA_PROYECTO / "datos"
RUTA_RESULTADOS = RUTA_PROYECTO / "resultados"
RUTA_MODELOS = RUTA_PROYECTO / "modelos"
RUTA_EVIDENCIAS = RUTA_PROYECTO / "evidencias"

RUTA_DATASET_MODELADO_90D = (
    RUTA_RESULTADOS
    / "dataset_modelado_90d.pkl"
)

RUTA_COLUMNAS_176 = (
    RUTA_RESULTADOS
    / "columnas_predictoras_176_90d.csv"
)


# ---------------------------------------------------------
# Comprobación de las rutas necesarias
# ---------------------------------------------------------

rutas_necesarias = {
    "proyecto": RUTA_PROYECTO,
    "datos": RUTA_DATOS,
    "resultados": RUTA_RESULTADOS,
    "modelos": RUTA_MODELOS,
    "evidencias": RUTA_EVIDENCIAS,
    "dataset_modelado": RUTA_DATASET_MODELADO_90D,
    "inventario_variables": RUTA_COLUMNAS_176
}

comprobacion_rutas = pd.DataFrame({
    "elemento": rutas_necesarias.keys(),
    "ruta": [
        str(ruta)
        for ruta in rutas_necesarias.values()
    ],
    "existe": [
        ruta.exists()
        for ruta in rutas_necesarias.values()
    ]
})

display(comprobacion_rutas)

print(
    "¿Están disponibles todos los elementos necesarios?",
    comprobacion_rutas["existe"].all()
)

## Carga y comprobación inicial del dataset de modelado

Se carga el dataset preparado en la fase anterior junto con el inventario de las 176 variables predictoras.

Antes de iniciar el análisis exploratorio, se comprueba que los datos conservan la estructura esperada. Para ello, se verifican las dimensiones del dataset, la unicidad de `customer_id`, la ausencia de valores nulos, la correspondencia con el inventario de variables predictoras y la coherencia de las particiones de entrenamiento y prueba.

Esta validación funciona como punto de control entre la preparación de los datos y su análisis estadístico, evitando trabajar con un archivo incompleto, desactualizado o diferente del validado al finalizar el notebook anterior.

Se espera obtener un dataset con 65.005 clientes y 179 columnas: 176 variables predictoras y las columnas adicionales `customer_id`, `churn` y `particion`.

In [ ]:
# ---------------------------------------------------------
# Carga del dataset y del inventario de variables
# ---------------------------------------------------------

dataset_modelado = pd.read_pickle(
    RUTA_DATASET_MODELADO_90D
)

inventario_variables = pd.read_csv(
    RUTA_COLUMNAS_176
)

columnas_predictoras = (
    inventario_variables["variable"]
    .tolist()
)


# ---------------------------------------------------------
# Comprobaciones estructurales
# ---------------------------------------------------------

comprobaciones_dataset = {
    "filas_correctas": (
        dataset_modelado.shape[0] == 65_005
    ),
    "columnas_totales_correctas": (
        dataset_modelado.shape[1] == 179
    ),
    "variables_predictoras_correctas": (
        len(columnas_predictoras) == 176
    ),
    "customer_id_unico": (
        dataset_modelado[
            "customer_id"
        ].nunique() == 65_005
    ),
    "sin_customer_id_duplicados": (
        dataset_modelado[
            "customer_id"
        ].duplicated().sum() == 0
    ),
    "sin_valores_nulos": (
        dataset_modelado
        .isna()
        .sum()
        .sum() == 0
    ),
    "columnas_predictoras_presentes": (
        set(columnas_predictoras)
        .issubset(
            dataset_modelado.columns
        )
    ),
    "churn_presente": (
        "churn"
        in dataset_modelado.columns
    ),
    "particion_presente": (
        "particion"
        in dataset_modelado.columns
    )
}

comprobaciones_dataset_df = pd.DataFrame({
    "comprobacion": comprobaciones_dataset.keys(),
    "resultado": comprobaciones_dataset.values()
})

display(comprobaciones_dataset_df)


# ---------------------------------------------------------
# Resumen general
# ---------------------------------------------------------

print(
    "Dimensión del dataset:",
    dataset_modelado.shape
)

print(
    "Número de variables predictoras:",
    len(columnas_predictoras)
)

print(
    "Clientes únicos:",
    dataset_modelado[
        "customer_id"
    ].nunique()
)

print(
    "Valores nulos totales:",
    dataset_modelado
    .isna()
    .sum()
    .sum()
)

print(
    "\nDistribución por partición:"
)

display(
    dataset_modelado[
        "particion"
    ]
    .value_counts()
    .rename_axis("particion")
    .reset_index(name="clientes")
)

print(
    "¿El dataset está listo para el análisis exploratorio?",
    all(
        comprobaciones_dataset.values()
    )
)

## Distribución de la variable objetivo

La variable objetivo `churn` indica si un cliente permanece o abandona. Conocer su distribución antes del modelado permite identificar posibles desequilibrios entre clases, que pueden afectar a la interpretación de métricas como la exactitud.

Se calculan las frecuencias absoluta y relativa de cada clase en el dataset completo y en las particiones de entrenamiento y prueba por separado. Con este análisis se comprueba:

1. Si existe un desequilibrio entre las clases de la variable objetivo.
2. Si la proporción de `churn` es similar en ambos conjuntos.
3. Si las particiones originales conservan una distribución coherente de la variable objetivo.

Una distribución semejante entre entrenamiento y prueba reduce el riesgo de que la evaluación final esté condicionada por una proporción de clases muy diferente de la observada durante el entrenamiento.

En esta etapa, el conjunto de prueba se utiliza únicamente para describir la composición de las particiones. Sus datos no intervienen en la selección de variables, el ajuste de hiperparámetros ni la toma de decisiones sobre los modelos.

In [ ]:
# ---------------------------------------------------------
# Distribución global de churn
# ---------------------------------------------------------

distribucion_global_churn = (
    dataset_modelado["churn"]
    .value_counts()
    .sort_index()
    .rename_axis("churn")
    .reset_index(name="clientes")
)

distribucion_global_churn["porcentaje"] = (
    distribucion_global_churn["clientes"]
    / distribucion_global_churn["clientes"].sum()
    * 100
)

print("Distribución global de churn:")

display(
    distribucion_global_churn
    .style
    .format({
        "clientes": "{:,.0f}",
        "porcentaje": "{:.2f}%"
    })
)


# ---------------------------------------------------------
# Distribución de churn por partición
# ---------------------------------------------------------

distribucion_churn_particion = (
    dataset_modelado
    .groupby(
        ["particion", "churn"],
        observed=True
    )
    .size()
    .rename("clientes")
    .reset_index()
)

distribucion_churn_particion[
    "porcentaje_particion"
] = (
    distribucion_churn_particion["clientes"]
    / distribucion_churn_particion
        .groupby("particion")["clientes"]
        .transform("sum")
    * 100
)

print("Distribución de churn por partición:")

display(
    distribucion_churn_particion
    .style
    .format({
        "clientes": "{:,.0f}",
        "porcentaje_particion": "{:.2f}%"
    })
)


# ---------------------------------------------------------
# Tasas de abandono
# ---------------------------------------------------------

tasa_churn_global = (
    dataset_modelado["churn"].mean()
)

tasa_churn_train = (
    dataset_modelado.loc[
        dataset_modelado["particion"] == "train",
        "churn"
    ].mean()
)

tasa_churn_test = (
    dataset_modelado.loc[
        dataset_modelado["particion"] == "test",
        "churn"
    ].mean()
)

diferencia_tasas = abs(
    tasa_churn_train
    - tasa_churn_test
)

print(
    "Tasa global de churn:",
    f"{tasa_churn_global:.2%}"
)

print(
    "Tasa de churn en entrenamiento:",
    f"{tasa_churn_train:.2%}"
)

print(
    "Tasa de churn en prueba:",
    f"{tasa_churn_test:.2%}"
)

print(
    "Diferencia absoluta entre particiones:",
    f"{diferencia_tasas:.4%}"
)

distribucion_objetivo_coherente = (
    set(
        dataset_modelado["churn"].unique()
    ) == {0, 1}
    and diferencia_tasas < 0.01
)

print(
    "¿La distribución de churn es coherente "
    "entre entrenamiento y prueba?",
    distribucion_objetivo_coherente
)

## Separación operativa de las particiones

Aunque el dataset reúne los registros de entrenamiento y prueba, ambas particiones se mantienen separadas durante el análisis y el modelado.

El conjunto de entrenamiento se utiliza para analizar distribuciones, identificar variables problemáticas, comparar modelos y ajustar hiperparámetros. El conjunto de prueba permanece reservado hasta la evaluación final del modelo seleccionado.

Esta separación evita posibles fugas de información, ya que cualquier decisión basada en los datos de prueba podría sesgar la estimación del rendimiento. Por tanto, incluso las operaciones descriptivas que puedan influir en el modelado, como seleccionar variables según su relación con `churn`, se realizan exclusivamente sobre el conjunto de entrenamiento.

En este bloque se construyen:

- `datos_train`, con los registros asignados al conjunto de entrenamiento.
- `datos_test`, con los registros reservados para la evaluación final.
- `X_train` y `X_test`, con las 176 variables predictoras.
- `y_train` y `y_test`, con la variable objetivo correspondiente a cada conjunto.

Finalmente, se comprueba que no existen clientes compartidos entre las particiones y que ambas matrices predictoras contienen exactamente las mismas columnas en el mismo orden.

In [ ]:
# ---------------------------------------------------------
# Separación de las particiones
# ---------------------------------------------------------

datos_train = (
    dataset_modelado.loc[
        dataset_modelado["particion"] == "train"
    ]
    .copy()
)

datos_test = (
    dataset_modelado.loc[
        dataset_modelado["particion"] == "test"
    ]
    .copy()
)


# ---------------------------------------------------------
# Matrices predictoras y variables objetivo
# ---------------------------------------------------------

X_train = datos_train[
    columnas_predictoras
].copy()

y_train = datos_train[
    "churn"
].copy()

X_test = datos_test[
    columnas_predictoras
].copy()

y_test = datos_test[
    "churn"
].copy()


# ---------------------------------------------------------
# Comprobación de clientes compartidos
# ---------------------------------------------------------

clientes_train = set(
    datos_train["customer_id"]
)

clientes_test = set(
    datos_test["customer_id"]
)

clientes_compartidos = (
    clientes_train
    .intersection(clientes_test)
)


# ---------------------------------------------------------
# Validaciones
# ---------------------------------------------------------

comprobaciones_particiones = {
    "filas_train_correctas": (
        len(datos_train) == 52_004
    ),
    "filas_test_correctas": (
        len(datos_test) == 13_001
    ),
    "variables_train_correctas": (
        X_train.shape == (52_004, 176)
    ),
    "variables_test_correctas": (
        X_test.shape == (13_001, 176)
    ),
    "mismas_columnas": (
        X_train.columns.tolist()
        == X_test.columns.tolist()
    ),
    "sin_clientes_compartidos": (
        len(clientes_compartidos) == 0
    ),
    "sin_nulos_train": (
        X_train.isna().sum().sum() == 0
    ),
    "sin_nulos_test": (
        X_test.isna().sum().sum() == 0
    ),
    "objetivo_train_binario": (
        set(y_train.unique()) == {0, 1}
    ),
    "objetivo_test_binario": (
        set(y_test.unique()) == {0, 1}
    )
}

comprobaciones_particiones_df = pd.DataFrame({
    "comprobacion": comprobaciones_particiones.keys(),
    "resultado": comprobaciones_particiones.values()
})

display(comprobaciones_particiones_df)


# ---------------------------------------------------------
# Resumen de dimensiones
# ---------------------------------------------------------

print(
    "Dimensión de X_train:",
    X_train.shape
)

print(
    "Dimensión de y_train:",
    y_train.shape
)

print(
    "Dimensión de X_test:",
    X_test.shape
)

print(
    "Dimensión de y_test:",
    y_test.shape
)

print(
    "Clientes compartidos entre particiones:",
    len(clientes_compartidos)
)

print(
    "¿Las particiones están correctamente separadas?",
    all(
        comprobaciones_particiones.values()
    )
)

## Revisión de los tipos de datos y la variabilidad de las variables predictoras

Antes de analizar relaciones estadísticas o entrenar modelos, se comprueba que las 176 variables predictoras presentan una estructura numérica coherente y una variabilidad suficiente dentro del conjunto de entrenamiento.

Esta revisión se realiza exclusivamente sobre `X_train`, ya que cualquier decisión relacionada con la conservación o exclusión de variables debe basarse únicamente en los datos de entrenamiento.

En este bloque se comprueba:

1. Que todas las variables predictoras sean numéricas.
2. Que no existan valores nulos, infinitos ni otros valores no finitos.
3. Que ninguna variable sea constante en el conjunto de entrenamiento.
4. Que no existan variables con una variabilidad prácticamente nula.

Las variables constantes no permiten diferenciar entre clientes. Por su parte, una varianza extremadamente baja puede indicar una capacidad discriminativa limitada y requerir una revisión posterior, aunque en esta fase no se excluyen variables automáticamente por este criterio.

Además, se genera un diagnóstico que recoge el tipo de dato, el número de valores únicos, el mínimo, el máximo, la media, la desviación estándar y la varianza de cada variable. Este inventario permite detectar posibles anomalías y documenta el estado del dataset antes del modelado.

In [ ]:
# ---------------------------------------------------------
# Comprobación de tipos numéricos
# ---------------------------------------------------------

columnas_no_numericas = (
    X_train
    .select_dtypes(exclude=[np.number])
    .columns
    .tolist()
)


# ---------------------------------------------------------
# Comprobación de valores finitos
# ---------------------------------------------------------

valores_train = X_train.to_numpy()

todos_valores_finitos = (
    np.isfinite(valores_train).all()
)


# ---------------------------------------------------------
# Diagnóstico de variabilidad
# ---------------------------------------------------------

diagnostico_variabilidad = pd.DataFrame({
    "variable": X_train.columns,
    "tipo_dato": [
        str(X_train[columna].dtype)
        for columna in X_train.columns
    ],
    "valores_unicos": [
        X_train[columna].nunique(
            dropna=False
        )
        for columna in X_train.columns
    ],
    "minimo": [
        X_train[columna].min()
        for columna in X_train.columns
    ],
    "maximo": [
        X_train[columna].max()
        for columna in X_train.columns
    ],
    "media": [
        X_train[columna].mean()
        for columna in X_train.columns
    ],
    "desviacion_estandar": [
        X_train[columna].std(ddof=0)
        for columna in X_train.columns
    ],
    "varianza": [
        X_train[columna].var(ddof=0)
        for columna in X_train.columns
    ]
})

diagnostico_variabilidad["constante"] = (
    diagnostico_variabilidad[
        "valores_unicos"
    ] == 1
)

UMBRAL_VARIANZA_BAJA = 1e-12

diagnostico_variabilidad[
    "varianza_muy_baja"
] = (
    diagnostico_variabilidad["varianza"]
    <= UMBRAL_VARIANZA_BAJA
)

variables_constantes_train = (
    diagnostico_variabilidad.loc[
        diagnostico_variabilidad["constante"],
        "variable"
    ]
    .tolist()
)

variables_varianza_muy_baja = (
    diagnostico_variabilidad.loc[
        diagnostico_variabilidad[
            "varianza_muy_baja"
        ],
        "variable"
    ]
    .tolist()
)


# ---------------------------------------------------------
# Resumen de resultados
# ---------------------------------------------------------

print(
    "Variables predictoras revisadas:",
    len(X_train.columns)
)

print(
    "Variables no numéricas:",
    len(columnas_no_numericas)
)

print(
    "Variables constantes en entrenamiento:",
    len(variables_constantes_train)
)

print(
    "Variables con varianza muy baja:",
    len(variables_varianza_muy_baja)
)

print(
    "¿Todos los valores son finitos?",
    todos_valores_finitos
)

print(
    "¿Todas las variables son numéricas?",
    len(columnas_no_numericas) == 0
)

print(
    "¿Existen variables constantes?",
    len(variables_constantes_train) > 0
)


# ---------------------------------------------------------
# Visualización del diagnóstico
# ---------------------------------------------------------

display(
    diagnostico_variabilidad.head(10)
)

print(
    "\nVariables con menor varianza:"
)

display(
    diagnostico_variabilidad[
        [
            "variable",
            "valores_unicos",
            "varianza",
            "constante",
            "varianza_muy_baja"
        ]
    ]
    .sort_values(
        "varianza",
        ascending=True
    )
    .head(15)
)


# ---------------------------------------------------------
# Validación global
# ---------------------------------------------------------

estructura_numerica_valida = (
    len(X_train.columns) == 176
    and len(columnas_no_numericas) == 0
    and todos_valores_finitos
    and len(variables_constantes_train) == 0
)

print(
    "¿La estructura numérica del conjunto de entrenamiento "
    "es válida?",
    estructura_numerica_valida
)

## Distribución, concentración de ceros y valores extremos

Las variables de actividad suelen presentar distribuciones asimétricas: muchos clientes registran valores bajos o nulos, mientras que un grupo reducido concentra niveles de uso elevados. Esta estructura es habitual en datos de actividad y debe analizarse antes del modelado.

Este bloque describe la distribución de cada variable predictora utilizando exclusivamente el conjunto de entrenamiento. Para cada variable se calculan:

- El porcentaje de valores iguales a cero.
- Los percentiles 1, 25, 50, 75, 95 y 99.
- El valor máximo.
- La media y la desviación estándar.
- El coeficiente de asimetría.
- El número y la proporción de valores potencialmente extremos según el criterio intercuartílico.

Este criterio considera potencialmente extremo cualquier valor situado por debajo de `Q1 - 1,5 × IQR` o por encima de `Q3 + 1,5 × IQR`. La identificación tiene una finalidad exclusivamente descriptiva: estos valores no se eliminan automáticamente, ya que una actividad elevada puede ser real y contener información relevante sobre el comportamiento del cliente.

Finalmente, se identifican las variables con mayor concentración de ceros, asimetría y proporción de valores extremos. Estos resultados permiten valorar posteriormente la conveniencia de aplicar transformaciones, escalado o métodos robustos durante el modelado.

In [ ]:
# ---------------------------------------------------------
# Estadísticos descriptivos de las variables de entrenamiento
# ---------------------------------------------------------

percentiles_train = X_train.quantile(
    [0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

percentiles_train.columns = [
    "percentil_01",
    "percentil_25",
    "mediana",
    "percentil_75",
    "percentil_95",
    "percentil_99"
]

diagnostico_distribuciones = pd.DataFrame({
    "variable": X_train.columns,
    "media": X_train.mean().values,
    "desviacion_estandar": (
        X_train.std(ddof=0).values
    ),
    "minimo": X_train.min().values,
    "maximo": X_train.max().values,
    "porcentaje_ceros": (
        X_train.eq(0).mean().values * 100
    ),
    "asimetria": X_train.skew().values
})

diagnostico_distribuciones = (
    diagnostico_distribuciones
    .set_index("variable")
    .join(percentiles_train)
    .reset_index()
)


# ---------------------------------------------------------
# Detección descriptiva de valores extremos
# ---------------------------------------------------------

q1 = X_train.quantile(0.25)
q3 = X_train.quantile(0.75)

rango_intercuartil = q3 - q1

limite_inferior = (
    q1 - 1.5 * rango_intercuartil
)

limite_superior = (
    q3 + 1.5 * rango_intercuartil
)

mascara_extremos = (
    X_train.lt(
        limite_inferior,
        axis=1
    )
    | X_train.gt(
        limite_superior,
        axis=1
    )
)

numero_extremos = (
    mascara_extremos.sum(axis=0)
)

porcentaje_extremos = (
    mascara_extremos.mean(axis=0) * 100
)

diagnostico_distribuciones[
    "numero_valores_extremos"
] = (
    diagnostico_distribuciones["variable"]
    .map(numero_extremos)
)

diagnostico_distribuciones[
    "porcentaje_valores_extremos"
] = (
    diagnostico_distribuciones["variable"]
    .map(porcentaje_extremos)
)


# ---------------------------------------------------------
# Presentación de resultados principales
# ---------------------------------------------------------

print(
    "Variables analizadas:",
    len(diagnostico_distribuciones)
)

print(
    "Variables con algún valor cero:",
    (
        diagnostico_distribuciones[
            "porcentaje_ceros"
        ] > 0
    ).sum()
)

print(
    "Variables con asimetría absoluta superior a 1:",
    (
        diagnostico_distribuciones[
            "asimetria"
        ].abs() > 1
    ).sum()
)

print(
    "Variables con algún valor extremo según IQR:",
    (
        diagnostico_distribuciones[
            "numero_valores_extremos"
        ] > 0
    ).sum()
)


print(
    "\nVariables con mayor porcentaje de ceros:"
)

display(
    diagnostico_distribuciones[
        [
            "variable",
            "porcentaje_ceros",
            "mediana",
            "percentil_95",
            "percentil_99",
            "maximo"
        ]
    ]
    .sort_values(
        "porcentaje_ceros",
        ascending=False
    )
    .head(15)
)


print(
    "\nVariables con mayor asimetría:"
)

display(
    diagnostico_distribuciones[
        [
            "variable",
            "media",
            "mediana",
            "percentil_99",
            "maximo",
            "asimetria"
        ]
    ]
    .assign(
        asimetria_absoluta=lambda tabla:
        tabla["asimetria"].abs()
    )
    .sort_values(
        "asimetria_absoluta",
        ascending=False
    )
    .head(15)
    .drop(
        columns="asimetria_absoluta"
    )
)


print(
    "\nVariables con mayor proporción de valores extremos:"
)

display(
    diagnostico_distribuciones[
        [
            "variable",
            "porcentaje_valores_extremos",
            "percentil_75",
            "percentil_99",
            "maximo"
        ]
    ]
    .sort_values(
        "porcentaje_valores_extremos",
        ascending=False
    )
    .head(15)
)


# ---------------------------------------------------------
# Validación del diagnóstico
# ---------------------------------------------------------

diagnostico_distribuciones_valido = (
    len(diagnostico_distribuciones) == 176
    and diagnostico_distribuciones[
        "variable"
    ].nunique() == 176
    and diagnostico_distribuciones
        .isna()
        .sum()
        .sum() == 0
    and np.isfinite(
        diagnostico_distribuciones[
            [
                "media",
                "desviacion_estandar",
                "minimo",
                "maximo",
                "porcentaje_ceros",
                "asimetria",
                "percentil_01",
                "percentil_25",
                "mediana",
                "percentil_75",
                "percentil_95",
                "percentil_99",
                "numero_valores_extremos",
                "porcentaje_valores_extremos"
            ]
        ].to_numpy()
    ).all()
)

print(
    "¿El diagnóstico de distribuciones es válido?",
    diagnostico_distribuciones_valido
)

## Relación individual entre las variables predictoras y `churn`

Una vez revisada la estructura general de las variables predictoras, se analiza su relación individual con la variable objetivo `churn`. Este análisis se realiza exclusivamente sobre el conjunto de entrenamiento para evitar que la información del conjunto de prueba influya en decisiones posteriores.

El análisis es univariante: cada variable se estudia por separado, sin considerar todavía sus interacciones con el resto. Para cada variable predictora se calculan:

- La media y la mediana de los clientes que permanecen.
- La media y la mediana de los clientes que abandonan.
- La diferencia entre las medias de ambos grupos.
- La diferencia estandarizada de medias.
- La correlación con la variable objetivo binaria `churn`.

La diferencia estandarizada permite comparar variables medidas en escalas distintas. Su signo indica la dirección de la asociación:

- Un valor positivo indica que la variable tiende a ser mayor entre los clientes que abandonan.
- Un valor negativo indica que tiende a ser mayor entre los clientes que permanecen.
- Un valor próximo a cero indica que existe poca diferencia individual entre ambos grupos.

La correlación se utiliza como una medida descriptiva de asociación lineal. Un valor reducido no implica necesariamente que una variable carezca de utilidad predictiva, ya que su relación con `churn` puede ser no lineal o depender de su interacción con otras variables.

Los resultados de este análisis no se utilizan para excluir variables en esta etapa. Su finalidad es identificar las variables que presentan las diferencias individuales más destacadas y facilitar la interpretación posterior de los modelos.

In [ ]:
# ---------------------------------------------------------
# Separación de las variables según la clase de churn
# ---------------------------------------------------------

X_train_churn_0 = X_train.loc[
    y_train == 0
]

X_train_churn_1 = X_train.loc[
    y_train == 1
]


# ---------------------------------------------------------
# Estadísticos por grupo
# ---------------------------------------------------------

media_churn_0 = X_train_churn_0.mean()
media_churn_1 = X_train_churn_1.mean()

mediana_churn_0 = X_train_churn_0.median()
mediana_churn_1 = X_train_churn_1.median()

varianza_churn_0 = X_train_churn_0.var(ddof=1)
varianza_churn_1 = X_train_churn_1.var(ddof=1)

n_churn_0 = len(X_train_churn_0)
n_churn_1 = len(X_train_churn_1)


# ---------------------------------------------------------
# Diferencia estandarizada de medias
# ---------------------------------------------------------

varianza_combinada = (
    (
        (n_churn_0 - 1) * varianza_churn_0
        + (n_churn_1 - 1) * varianza_churn_1
    )
    / (
        n_churn_0
        + n_churn_1
        - 2
    )
)

desviacion_combinada = np.sqrt(
    varianza_combinada
)

diferencia_medias = (
    media_churn_1
    - media_churn_0
)

diferencia_estandarizada = (
    diferencia_medias
    / desviacion_combinada.replace(0, np.nan)
)


# ---------------------------------------------------------
# Correlación de cada variable con churn
# ---------------------------------------------------------

datos_correlacion = X_train.copy()

datos_correlacion["churn"] = (
    y_train.to_numpy()
)

correlacion_con_churn = (
    datos_correlacion
    .corr(numeric_only=True)["churn"]
    .drop("churn")
)


# ---------------------------------------------------------
# Construcción del diagnóstico univariante
# ---------------------------------------------------------

diagnostico_relacion_churn = pd.DataFrame({
    "variable": X_train.columns,
    "media_churn_0": (
        media_churn_0
        .reindex(X_train.columns)
        .to_numpy()
    ),
    "media_churn_1": (
        media_churn_1
        .reindex(X_train.columns)
        .to_numpy()
    ),
    "mediana_churn_0": (
        mediana_churn_0
        .reindex(X_train.columns)
        .to_numpy()
    ),
    "mediana_churn_1": (
        mediana_churn_1
        .reindex(X_train.columns)
        .to_numpy()
    ),
    "diferencia_medias": (
        diferencia_medias
        .reindex(X_train.columns)
        .to_numpy()
    ),
    "diferencia_estandarizada": (
        diferencia_estandarizada
        .reindex(X_train.columns)
        .to_numpy()
    ),
    "correlacion_churn": (
        correlacion_con_churn
        .reindex(X_train.columns)
        .to_numpy()
    )
})

diagnostico_relacion_churn[
    "diferencia_estandarizada_absoluta"
] = (
    diagnostico_relacion_churn[
        "diferencia_estandarizada"
    ].abs()
)

diagnostico_relacion_churn[
    "correlacion_absoluta"
] = (
    diagnostico_relacion_churn[
        "correlacion_churn"
    ].abs()
)


# ---------------------------------------------------------
# Presentación de resultados
# ---------------------------------------------------------

print(
    "Clientes sin churn en entrenamiento:",
    f"{n_churn_0:,}"
)

print(
    "Clientes con churn en entrenamiento:",
    f"{n_churn_1:,}"
)

print(
    "Variables analizadas:",
    len(diagnostico_relacion_churn)
)


print(
    "\nVariables con mayor diferencia estandarizada:"
)

display(
    diagnostico_relacion_churn[
        [
            "variable",
            "media_churn_0",
            "media_churn_1",
            "diferencia_medias",
            "diferencia_estandarizada",
            "correlacion_churn"
        ]
    ]
    .sort_values(
        "diferencia_estandarizada",
        key=lambda serie: serie.abs(),
        ascending=False
    )
    .head(20)
)


print(
    "\nVariables con mayor correlación absoluta con churn:"
)

display(
    diagnostico_relacion_churn[
        [
            "variable",
            "mediana_churn_0",
            "mediana_churn_1",
            "correlacion_churn",
            "diferencia_estandarizada"
        ]
    ]
    .sort_values(
        "correlacion_churn",
        key=lambda serie: serie.abs(),
        ascending=False
    )
    .head(20)
)


# ---------------------------------------------------------
# Validación del diagnóstico
# ---------------------------------------------------------

columnas_numericas_diagnostico = [
    "media_churn_0",
    "media_churn_1",
    "mediana_churn_0",
    "mediana_churn_1",
    "diferencia_medias",
    "diferencia_estandarizada",
    "correlacion_churn"
]

diagnostico_relacion_valido = (
    len(diagnostico_relacion_churn) == 176
    and diagnostico_relacion_churn[
        "variable"
    ].nunique() == 176
    and diagnostico_relacion_churn[
        columnas_numericas_diagnostico
    ].isna().sum().sum() == 0
    and np.isfinite(
        diagnostico_relacion_churn[
            columnas_numericas_diagnostico
        ].to_numpy()
    ).all()
)

print(
    "¿El diagnóstico de relación con churn es válido?",
    diagnostico_relacion_valido
)

## Correlación y redundancia entre las variables predictoras

Muchas de las 176 variables se han construido a partir de los mismos perfiles de actividad y de resúmenes temporales relacionados. Por este motivo, es esperable que algunas presenten una correlación elevada entre sí.

Este bloque identifica posibles redundancias utilizando exclusivamente `X_train`, mientras que el conjunto de prueba permanece separado.

Se calcula la matriz de correlaciones de Pearson entre las variables predictoras y se identifican los pares cuya correlación absoluta supera diferentes umbrales.

Una correlación elevada no implica que alguna de las variables deba excluirse automáticamente, ya que dos variables estrechamente relacionadas pueden aportar información diferente a determinados modelos. No obstante, este diagnóstico permite:

- Detectar variables prácticamente equivalentes.
- Identificar posibles problemas de multicolinealidad en modelos lineales.
- Comprender la estructura interna del conjunto de variables predictoras.
- Documentar qué variables contienen información similar.

Para evitar duplicidades, solo se analiza el triángulo superior de la matriz. De esta manera, cada par de variables aparece una única vez y se excluyen las correlaciones de cada variable consigo misma.

Como resultado, se obtiene una tabla ordenada con los pares que presentan una mayor correlación absoluta y un resumen del número de pares que superan los umbrales de `0,90`, `0,95` y `0,99`. En esta fase no se excluye ninguna variable.

In [ ]:
# ---------------------------------------------------------
# Cálculo de la matriz de correlaciones en entrenamiento
# ---------------------------------------------------------

matriz_correlaciones = X_train.corr(
    method="pearson"
)


# ---------------------------------------------------------
# Selección del triángulo superior
# ---------------------------------------------------------

mascara_triangulo_superior = np.triu(
    np.ones(
        matriz_correlaciones.shape,
        dtype=bool
    ),
    k=1
)

correlaciones_pares = (
    matriz_correlaciones
    .where(mascara_triangulo_superior)
    .stack()
    .reset_index()
)

correlaciones_pares.columns = [
    "variable_1",
    "variable_2",
    "correlacion"
]

correlaciones_pares[
    "correlacion_absoluta"
] = (
    correlaciones_pares["correlacion"]
    .abs()
)

correlaciones_pares = (
    correlaciones_pares
    .sort_values(
        "correlacion_absoluta",
        ascending=False
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Conteo por umbrales de correlación
# ---------------------------------------------------------

umbrales_correlacion = [
    0.90,
    0.95,
    0.99
]

resumen_correlaciones_altas = pd.DataFrame({
    "umbral_absoluto": umbrales_correlacion,
    "numero_pares": [
        (
            correlaciones_pares[
                "correlacion_absoluta"
            ] >= umbral
        ).sum()
        for umbral in umbrales_correlacion
    ]
})


# ---------------------------------------------------------
# Presentación de resultados
# ---------------------------------------------------------

print(
    "Dimensión de la matriz de correlaciones:",
    matriz_correlaciones.shape
)

print(
    "Número total de pares analizados:",
    len(correlaciones_pares)
)

print(
    "\nNúmero de pares por umbral:"
)

display(
    resumen_correlaciones_altas
)

print(
    "\nVeinte pares con mayor correlación absoluta:"
)

display(
    correlaciones_pares.head(20)
)


# ---------------------------------------------------------
# Variables implicadas en correlaciones muy altas
# ---------------------------------------------------------

pares_correlacion_099 = (
    correlaciones_pares.loc[
        correlaciones_pares[
            "correlacion_absoluta"
        ] >= 0.99
    ]
    .copy()
)

variables_en_pares_099 = sorted(
    set(
        pares_correlacion_099["variable_1"]
    )
    .union(
        pares_correlacion_099["variable_2"]
    )
)

print(
    "Pares con correlación absoluta >= 0.99:",
    len(pares_correlacion_099)
)

print(
    "Variables implicadas en esos pares:",
    len(variables_en_pares_099)
)


# ---------------------------------------------------------
# Validación del diagnóstico
# ---------------------------------------------------------

numero_pares_esperado = (
    len(columnas_predictoras)
    * (len(columnas_predictoras) - 1)
    // 2
)

diagnostico_correlaciones_valido = (
    matriz_correlaciones.shape == (176, 176)
    and len(correlaciones_pares)
        == numero_pares_esperado
    and correlaciones_pares[
        "variable_1"
    ].notna().all()
    and correlaciones_pares[
        "variable_2"
    ].notna().all()
    and np.isfinite(
        correlaciones_pares[
            [
                "correlacion",
                "correlacion_absoluta"
            ]
        ].to_numpy()
    ).all()
    and correlaciones_pares[
        "correlacion_absoluta"
    ].between(0, 1).all()
)

print(
    "¿El diagnóstico de correlaciones es válido?",
    diagnostico_correlaciones_valido
)

## Almacenamiento de los resultados del análisis exploratorio

El análisis realizado hasta este punto genera distintos diagnósticos sobre la estructura y el comportamiento de las variables predictoras. Para mantener la trazabilidad del proyecto, estos resultados se almacenan como archivos independientes en la carpeta `resultados`.

Se conservan los siguientes elementos:

- La distribución global de la variable objetivo `churn`.
- La distribución de `churn` en los conjuntos de entrenamiento y prueba.
- El diagnóstico de los tipos de datos y la variabilidad.
- El análisis de la concentración de ceros, la asimetría y los valores extremos.
- La relación univariante entre cada variable predictora y `churn`.
- Los pares de variables ordenados por su correlación absoluta.
- El resumen de los pares que superan los umbrales de correlación absoluta de `0,90`, `0,95` y `0,99`.

Estos archivos proporcionan evidencias reproducibles del análisis previo al modelado. Además, permiten consultar los resultados sin repetir los cálculos y facilitan la elaboración posterior de tablas, anexos y conclusiones para la memoria del proyecto.

Este almacenamiento no modifica el dataset de modelado ni implica la exclusión de variables. Su finalidad es exclusivamente analítica y documental.

In [ ]:
import json


# ---------------------------------------------------------
# Definición de las rutas de salida
# ---------------------------------------------------------

RUTA_DISTRIBUCION_CHURN_GLOBAL = (
    RUTA_RESULTADOS
    / "eda_distribucion_churn_global.csv"
)

RUTA_DISTRIBUCION_CHURN_PARTICION = (
    RUTA_RESULTADOS
    / "eda_distribucion_churn_particion.csv"
)

RUTA_DIAGNOSTICO_VARIABILIDAD = (
    RUTA_RESULTADOS
    / "eda_diagnostico_variabilidad.csv"
)

RUTA_DIAGNOSTICO_DISTRIBUCIONES = (
    RUTA_RESULTADOS
    / "eda_diagnostico_distribuciones.csv"
)

RUTA_RELACION_CHURN = (
    RUTA_RESULTADOS
    / "eda_relacion_univariante_churn.csv"
)

RUTA_CORRELACIONES_PARES = (
    RUTA_RESULTADOS
    / "eda_correlaciones_pares.csv"
)

RUTA_RESUMEN_CORRELACIONES = (
    RUTA_RESULTADOS
    / "eda_resumen_correlaciones_altas.csv"
)

RUTA_RESUMEN_EDA = (
    RUTA_RESULTADOS
    / "eda_resumen_general.json"
)


# ---------------------------------------------------------
# Almacenamiento de las tablas
# ---------------------------------------------------------

distribucion_global_churn.to_csv(
    RUTA_DISTRIBUCION_CHURN_GLOBAL,
    index=False
)

distribucion_churn_particion.to_csv(
    RUTA_DISTRIBUCION_CHURN_PARTICION,
    index=False
)

diagnostico_variabilidad.to_csv(
    RUTA_DIAGNOSTICO_VARIABILIDAD,
    index=False
)

diagnostico_distribuciones.to_csv(
    RUTA_DIAGNOSTICO_DISTRIBUCIONES,
    index=False
)

diagnostico_relacion_churn.to_csv(
    RUTA_RELACION_CHURN,
    index=False
)

correlaciones_pares.to_csv(
    RUTA_CORRELACIONES_PARES,
    index=False
)

resumen_correlaciones_altas.to_csv(
    RUTA_RESUMEN_CORRELACIONES,
    index=False
)


# ---------------------------------------------------------
# Construcción del resumen general
# ---------------------------------------------------------

resumen_eda = {
    "clientes_totales": int(
        len(dataset_modelado)
    ),
    "clientes_train": int(
        len(datos_train)
    ),
    "clientes_test": int(
        len(datos_test)
    ),
    "variables_predictoras": int(
        len(columnas_predictoras)
    ),
    "tasa_churn_global": float(
        tasa_churn_global
    ),
    "tasa_churn_train": float(
        tasa_churn_train
    ),
    "tasa_churn_test": float(
        tasa_churn_test
    ),
    "diferencia_tasa_churn_train_test": float(
        diferencia_tasas
    ),
    "variables_no_numericas": int(
        len(columnas_no_numericas)
    ),
    "variables_constantes_train": int(
        len(variables_constantes_train)
    ),
    "variables_varianza_muy_baja": int(
        len(variables_varianza_muy_baja)
    ),
    "variables_con_ceros": int(
        (
            diagnostico_distribuciones[
                "porcentaje_ceros"
            ] > 0
        ).sum()
    ),
    "variables_asimetria_absoluta_superior_1": int(
        (
            diagnostico_distribuciones[
                "asimetria"
            ].abs() > 1
        ).sum()
    ),
    "variables_con_extremos_iqr": int(
        (
            diagnostico_distribuciones[
                "numero_valores_extremos"
            ] > 0
        ).sum()
    ),
    "pares_correlacion_090": int(
        (
            correlaciones_pares[
                "correlacion_absoluta"
            ] >= 0.90
        ).sum()
    ),
    "pares_correlacion_095": int(
        (
            correlaciones_pares[
                "correlacion_absoluta"
            ] >= 0.95
        ).sum()
    ),
    "pares_correlacion_099": int(
        (
            correlaciones_pares[
                "correlacion_absoluta"
            ] >= 0.99
        ).sum()
    )
}

with open(
    RUTA_RESUMEN_EDA,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        resumen_eda,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Comprobación de los archivos generados
# ---------------------------------------------------------

archivos_eda = [
    RUTA_DISTRIBUCION_CHURN_GLOBAL,
    RUTA_DISTRIBUCION_CHURN_PARTICION,
    RUTA_DIAGNOSTICO_VARIABILIDAD,
    RUTA_DIAGNOSTICO_DISTRIBUCIONES,
    RUTA_RELACION_CHURN,
    RUTA_CORRELACIONES_PARES,
    RUTA_RESUMEN_CORRELACIONES,
    RUTA_RESUMEN_EDA
]

resumen_archivos_eda = pd.DataFrame({
    "archivo": [
        ruta.name
        for ruta in archivos_eda
    ],
    "existe": [
        ruta.exists()
        for ruta in archivos_eda
    ],
    "tamaño_mb": [
        round(
            ruta.stat().st_size / (1024 ** 2),
            3
        )
        if ruta.exists()
        else np.nan
        for ruta in archivos_eda
    ]
})

display(resumen_archivos_eda)

print(
    "Archivos de evidencias generados:",
    len(archivos_eda)
)

print(
    "¿Se guardaron todas las evidencias del EDA?",
    resumen_archivos_eda["existe"].all()
)

print(
    "Filas del diagnóstico de variabilidad:",
    pd.read_csv(
        RUTA_DIAGNOSTICO_VARIABILIDAD
    ).shape[0]
)

print(
    "Filas del diagnóstico de distribuciones:",
    pd.read_csv(
        RUTA_DIAGNOSTICO_DISTRIBUCIONES
    ).shape[0]
)

print(
    "Filas del análisis univariante:",
    pd.read_csv(
        RUTA_RELACION_CHURN
    ).shape[0]
)

print(
    "Pares de correlación almacenados:",
    pd.read_csv(
        RUTA_CORRELACIONES_PARES
    ).shape[0]
)